# Code 14.4.1: DPO does not stop; IPO stops at the target margin

A three-way softmax (chosen, rejected, other), $`\beta = 0.1`$, reference equal to the uniform initial policy. IPO's target gap is $`1/(2\beta) = 5`$ nats. DPO is run with a larger step size so that it is visible, on this convex toy, that the logistic loss is still increasing the gap after it has passed 5.


In [ ]:
import torch
import torch.nn.functional as F

def run(kind, steps, lr, beta=0.1):
    logits = torch.zeros(3, requires_grad=True)
    ref = F.log_softmax(logits.detach(), dim=0)
    opt = torch.optim.SGD([logits], lr=lr)
    for _ in range(steps):
        logp = F.log_softmax(logits, dim=0)
        h = (logp[0] - ref[0]) - (logp[1] - ref[1])
        loss = -F.logsigmoid(beta * h) if kind == "dpo" else (h - 1 / (2 * beta)) ** 2
        opt.zero_grad()
        loss.backward()
        opt.step()
    with torch.no_grad():
        logp = F.log_softmax(logits, dim=0)
        h = (logp[0] - ref[0]) - (logp[1] - ref[1])
        pi = logp.exp()
        print(f"{kind:4s}  pi {[round(p, 4) for p in pi.tolist()]}  h {h.item():.3f}  "
              f"target {1 / (2 * beta):.1f}")

run("dpo", steps=40, lr=2.0)
run("ipo", steps=80, lr=0.05)


**Expected output**

Output:

```text
dpo   pi [0.964, 0.0013, 0.0348]  h 6.644  target 5.0
ipo   pi [0.9184, 0.0062, 0.0754]  h 5.000  target 5.0
```
